# Post Pandemic Regime Shifts in Labor Market: Pulling Data from FRED and other sources

Federal Reserve Economic Data — St. Louis Fed
Atlanta Fed Wage Growth Tracker
Philadelphia Fed Real-Time Data Research Set
hiladelphia Fed Survey of Professional Forecasters
Dallas Fed Trimmed Mean PCE 
Kansas City Fed Labor Market Conditions Indicators

## Purpose

## Research Context

## Inputs and Outputs

## Imports and Configuration

In [7]:
%matplotlib inline

import re
import time
from pathlib import Path
import requests

import numpy as np
import pandas as pd

from fredapi import Fred

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [8]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_LaborTightnessRegimeShift"
# repo_root = Path(r"C:\Users\sumai\Documents\ML_LaborTightnessRegimeShift")

data_root = repo_root / "data"
raw_root  = data_root / "raw"
fred_root    = raw_root  / "fred"
external_root = raw_root / "external_sources"

fred_root.mkdir(parents=True, exist_ok=True)
external_root.mkdir(parents=True, exist_ok=True)

In [9]:
API_KEY = """
 keyissomewherehere
"""

api_key = re.sub(r"\s+", "", API_KEY)

fred = Fred(api_key=api_key)

start_date = "2000-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

In [10]:
series_map = {
    # job market
    "job_openings_level":         "JTSJOL",
    "hires_rate":                 "JTSHIR",
    "hires_level":                "JTSHIL",
    "quits_rate":                 "JTSQUR",
    "quits_level":                "JTSQUL",
    "total_separations_rate":     "JTSTSR",
    "total_separations_level":    "JTSTSL",
    "layoffs_discharges_level":   "JTSLDL",
    "layoffs_discharges_rate":    "JTSLDR",

    # labor force and unemployment 
    "unemployment_rate":          "UNRATE",
    "unemployment_level":         "UNEMPLOY",
    "u6_rate":                    "U6RATE",
    "prime_age_lfpr":             "LNS11300060",
    "lfpr":                       "CIVPART",
    "epop_ratio":                 "EMRATIO",
    "prime_age_urate":            "LNS13025703",
    "avg_weeks_unemployed":       "UEMPMEAN",

    # payrolls
    "payrolls_nonfarm":           "PAYEMS",
    "payrolls_private":           "USPRIV",
    "payrolls_manufacturing":     "MANEMP",
    "payrolls_services":          "SRVPRD",
    "payrolls_construction":      "USCONS",

    # rarnings, hours, and compensation
    "ahe_private":                "CES0500000003",
    "awe_private":                "CES0500000011",
    "awh_private":                "AWHAETP",
    "awe_manufacturing":          "CES3000000008",
    "ahe_manufacturing":          "CES3000000003",
    "eci_total":                  "ECIALLCIV",
    "eci_wages":                  "ECIWAG",

    # cpi
    "cpi_all":                    "CPIAUCSL",
    "cpi_core":                   "CPILFESL",
    "cpi_less_shelter":           "CUSR0000SA0L2",
    "cpi_services_less_rent":     "CUUR0000SASL2RS",
    "cpi_services_less_energy":   "CUSR0000SASLE",
    "cpi_shelter":                "CUSR0000SEHA",
    "cpi_medical":                "CPIMEDSL",
    "cpi_food":                   "CPIUFDSL",

    # PCE price
    "pce_price":                  "PCEPI",
    "pce_core":                   "PCEPILFE",
    "pce_trimmed_12m":            "PCETRIM12M159SFRBDAL",
    "pce_trimmed_1m":             "PCETRIM1M158SFRBDAL",
    "pce_services":               "DSERRG3M086SBEA",

    # income and spending 
    "saving_rate":                "PMSAVE",
    "real_dpi":                   "DSPIC96",
    "real_pi":                    "RPI",
    "real_pi_less_transfers":     "W875RX1",
    "retail_advance":             "RSAFS",
    "real_retail":                "RRSFS",

    # production
    "ip_total":                   "INDPRO",
    "ip_manufacturing":           "IPMAN",
    "capacity_util":              "CUMFNS",

    # housing
    "housing_starts":             "HOUST",
    "building_permits":           "PERMIT",
    "new_home_sales":             "HSN1F",

    # orders/consumer
    "consumer_sentiment":         "UMCSENT",
    "durable_orders":             "DGORDER",
    "capex_orders":               "NEWORDER",

    # credit
    "ci_loans":                   "BUSLOANS",
    "bank_credit":                "TOTBKCR",
    "m2":                         "M2SL",

    # rates and stuff
    "treasury_1m":                "DGS1MO",
    "treasury_3m":                "DGS3MO",
    "treasury_2y":                "DGS2",
    "treasury_10y":               "GS10",
    "fed_funds":                  "FEDFUNDS",
    "aaa_yield":                  "AAA",
    "baa_yield":                  "BAA",
    "hy_oas":                     "BAMLH0A0HYM2",
    "bbb_oas":                    "BAMLC0A4CBBB",

    # inflation expectations
    "breakeven_5y":               "T5YIE",
    "breakeven_10y":              "T10YIE",
    "forward_5y5y":               "T5YIFR",
    "michigan_1y":                "MICH",
    "michigan_5y":                "MICH5Y",

    # Kansas fed labor market conditions
    "kcfed_lmci_activity":        "FRBKCLMCILA",
    "kcfed_lmci_momentum":        "FRBKCLMCIM",

    # actual recession indicator
    "recession":                  "USREC",

    # other
    "output_per_hour":            "OPHNFB",
    "unit_labor_costs":           "ULCNFB",
}

print(f"Series Map: {len(series_map)}")

Series Map: 79


In [11]:
def fetch_with_retry(url, session=None, max_tries=5, timeout=120, backoff=2.0):
    
    session  = session or requests.Session()
    last_err = None
    
    for attempt in range(1, max_tries + 1):
        
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()
            return resp
            
        except Exception as exc:
            last_err = exc
            if attempt < max_tries:
                wait = backoff * attempt
                print(f"  Retry {attempt}/{max_tries} for {url} waiting {wait:.1f}s.")
                time.sleep(wait)
            
    raise RuntimeError(f"Download failed after {max_tries} attempts for {url}. Last error: {last_err}")


def pull_fred_series(series_id, start, end, max_tries=5, backoff=1.5):
    last_err = None
    
    for attempt in range(1, max_tries + 1):
        try:
            return fred.get_series(series_id, observation_start=start, observation_end=end)
        
        except Exception as exc:
            last_err = exc
            if attempt < max_tries:
                wait = backoff * attempt
                print(f"  Retry {attempt}/{max_tries} for series {series_id} — waiting {wait:.1f}s.")
                time.sleep(wait)
    
    raise RuntimeError(f"FRED pull failed for {series_id}. Last error: {last_err}")


def pull_fred_meta(series_id, max_tries=5, backoff=1.5):
    
    for attempt in range(1, max_tries + 1):
        try:
            return fred.get_series_info(series_id)
        
        except Exception as exc:
            if attempt < max_tries:
                wait = backoff * attempt
                print(f"  Retry {attempt}/{max_tries} for metadata {series_id} —waiting {wait:.1f}s.")
                time.sleep(wait)
            
            else:
                return pd.Series(dtype="object")